# Solutions — Classes: Dunder Methods, Inheritance, Composition, Registries

Exercise notebook: `05_functions_and_oop/04_classes_inheritance_composition.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · A native-feeling Batch container

**Working solution**

In [ ]:
class Batch:
    def __init__(self, records=None, batch_id="b0"):
        self.batch_id = batch_id
        self._records = list(records or [])

    def append(self, record):
        self._records.append(record)

    def __len__(self):
        return len(self._records)

    def __iter__(self):
        return iter(self._records)

    def __getitem__(self, index):
        if isinstance(index, slice):
            return Batch(self._records[index], self.batch_id)
        return self._records[index]

    def __eq__(self, other):
        return isinstance(other, Batch) and (self.batch_id, self._records) == (other.batch_id, other._records)

    def __repr__(self):
        return f"Batch(id={self.batch_id!r}, size={len(self)})"

**Explanation**

Python 'interfaces' are protocols of dunder methods. `__len__` also gives truthiness for free (`bool(b)` uses `__len__` when `__bool__`
is absent). `__getitem__` receives a `slice` object for `b[1:3]`.

**Complexity:** O(1) except slicing O(k).

**Common mistakes**

- Storing the caller's list without copying → external mutation leaks in.
- Returning a plain list from slicing (loses the type).

## Exercise 2 — 🟡 Medium · Template method with inheritance and super()

**Working solution**

In [ ]:
import csv
import json
from pathlib import Path

class FileSource:
    def __init__(self, path):
        self.path = Path(path)
        self.skipped = 0

    def parse(self, line_no, line):
        raise NotImplementedError

    def records(self):
        self.skipped = 0
        with self.path.open(encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                record = self.parse(line_no, line.rstrip("\n"))
                if record is None:
                    self.skipped += 1
                else:
                    yield record


class CsvSource(FileSource):
    def __init__(self, path, delimiter=","):
        super().__init__(path)
        self.delimiter = delimiter
        self._header = None

    def records(self):
        self._header = None
        yield from super().records()

    def parse(self, line_no, line):
        values = next(csv.reader([line], delimiter=self.delimiter))
        if self._header is None:
            self._header = values
            self.skipped -= 1          # the header is not a skipped row
            return None
        if len(values) != len(self._header):
            return None
        return dict(zip(self._header, values))


class JsonlSource(FileSource):
    def parse(self, line_no, line):
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            return None
        return obj if isinstance(obj, dict) else None

**Explanation**

The base class owns the algorithm (open, iterate, skip, count); subclasses supply one hook. `super().__init__(path)` initialises the
parent's state. `yield from super().records()` extends a parent generator. Note: in Python you could equally pass a `parse` *function* into
one class — choose inheritance when subclasses also need their own state (like the CSV header).

**Complexity:** O(lines).

**Common mistakes**

- Forgetting `super().__init__` → `AttributeError: 'CsvSource' object has no attribute 'path'`.
- Splitting CSV lines on `,` manually (quoted commas).

## Exercise 3 — 🟡 Medium · Composable pipelines with operators

**Working solution**

In [ ]:
class Pipeline:
    def __init__(self, *steps, name="pipeline"):
        self.steps = list(steps)
        self.name = name

    def add(self, step):
        self.steps.append(step)
        return self

    def run(self, records):
        for step in self.steps:
            records = step(records)
        return list(records)

    def __or__(self, other):
        if isinstance(other, Pipeline):
            return Pipeline(*self.steps, *other.steps, name=f"{self.name}+{other.name}")
        if callable(other):
            return Pipeline(*self.steps, other, name=self.name)
        return NotImplemented

    def __len__(self):
        return len(self.steps)

    def __repr__(self):
        return f"Pipeline({self.name!r}, steps={len(self)})"

**Explanation**

Composition: a pipeline *has* steps rather than *being* a subclass of some `AbstractStep`. Overloading `|` makes composition read like a shell
pipe (Apache Beam uses exactly this). Steps can be generator functions, so the whole chain stays lazy until `list()`.

**Complexity:** O(n × steps).

**Common mistakes**

- Mutating `self` inside `__or__` — operators should return new objects.

## Exercise 4 — 🟠 Hard · Plugin registry with __init_subclass__

**Working solution**

In [ ]:
class Transform:
    registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        name = cls.__dict__.get("name")
        if name is None:
            return
        if name in Transform.registry:
            raise ValueError(f"transform {name!r} already registered")
        Transform.registry[name] = cls

    @classmethod
    def create(cls, name, **params):
        try:
            klass = cls.registry[name]
        except KeyError:
            raise KeyError(f"unknown transform {name!r}; known: {sorted(cls.registry)}") from None
        return klass(**params)

    def apply(self, record: dict) -> dict:
        raise NotImplementedError

    def __call__(self, record):
        return self.apply(record)


class Rename(Transform):
    name = "rename"
    def __init__(self, mapping):
        self.mapping = mapping
    def apply(self, record):
        return {self.mapping.get(k, k): v for k, v in record.items()}


class Cast(Transform):
    name = "cast"
    _types = {"int": int, "float": float, "str": str}
    def __init__(self, column, to):
        self.column, self.fn = column, self._types[to]
    def apply(self, record):
        if record.get(self.column) is None:
            return dict(record)
        return {**record, self.column: self.fn(record[self.column])}


class Drop(Transform):
    name = "drop"
    def __init__(self, columns):
        self.columns = set(columns)
    def apply(self, record):
        return {k: v for k, v in record.items() if k not in self.columns}


def run_transforms(record, specs):
    for spec in specs:
        params = {k: v for k, v in spec.items() if k != "name"}
        record = Transform.create(spec["name"], **params)(record)
    return record

**Explanation**

`__init_subclass__` runs when a subclass is *defined* — a hook Java lacks without annotation processors/reflection scanning. Combined with
`create(name, **params)` it turns config (JSON/YAML) into objects. `cls.__dict__.get("name")` checks only the subclass's own attribute so
intermediate subclasses don't re-register their parent's name.

**Complexity:** O(1) lookup.

**Common mistakes**

- Using `getattr(cls, 'name')` — inherited names cause false duplicate errors.
- Keeping the registry on each subclass instead of the base.

## Challenge — 🔴 Very Hard · A mini immutable Table (DataFrame preview)

**Working solution**

In [ ]:
class Table:
    def __init__(self, rows=None):
        self._rows = [dict(r) for r in (rows or [])]

    @property
    def columns(self):
        return list(dict.fromkeys(k for r in self._rows for k in r))

    def __len__(self):
        return len(self._rows)

    def __iter__(self):
        return (dict(r) for r in self._rows)

    def __getitem__(self, column):
        return [r.get(column) for r in self._rows]

    def where(self, predicate):
        return Table(r for r in self._rows if predicate(r))

    def select(self, *cols):
        return Table({c: r.get(c) for c in cols} for r in self._rows)

    def order_by(self, col, desc=False):
        present = [r for r in self._rows if r.get(col) is not None]
        missing = [r for r in self._rows if r.get(col) is None]
        return Table(sorted(present, key=lambda r: r[col], reverse=desc) + missing)

    def with_column(self, name, func):
        return Table({**r, name: func(r)} for r in self._rows)

    def group_by(self, *keys):
        return _Grouped(self, keys)

    def join(self, other, on, how="inner"):
        if how not in ("inner", "left"):
            raise ValueError("how must be 'inner' or 'left'")
        index = {}
        for r in other._rows:
            index.setdefault(r.get(on), []).append(r)
        right_cols = [c for c in other.columns if c != on]
        out = []
        for left in self._rows:
            matches = index.get(left.get(on), [])
            if matches:
                out.extend({**left, **{c: m.get(c) for c in right_cols}} for m in matches)
            elif how == "left":
                out.append({**left, **{c: None for c in right_cols}})
        return Table(out)

    def __repr__(self):
        cols = self.columns
        preview = self._rows[:5]
        cells = [[str(c) for c in cols]] + [[str(r.get(c)) for c in cols] for r in preview]
        widths = [max(len(row[i]) for row in cells) for i in range(len(cols))]
        lines = ["  ".join(v.ljust(w) for v, w in zip(row, widths)) for row in cells]
        return "\n".join(lines + [f"[{len(self)} rows x {len(cols)} columns]"])


class _Grouped:
    def __init__(self, table, keys):
        self.table, self.keys = table, keys

    def agg(self, **aggs):
        groups = {}
        for r in self.table._rows:
            groups.setdefault(tuple(r.get(k) for k in self.keys), []).append(r)
        rows = []
        for key, members in groups.items():
            row = dict(zip(self.keys, key))
            for out_name, (column, func) in aggs.items():
                row[out_name] = func([m.get(column) for m in members])
            rows.append(row)
        return Table(rows)

**Explanation**

Immutability (every method returns a new `Table`) is how pandas method-chaining and Spark DataFrames feel: `t.where(...).group_by(...).agg(...)`.
A helper `_Grouped` object mirrors pandas' `GroupBy`/Spark's `GroupedData`. Hash join (index the right side, probe with the left) is the
same algorithm Spark uses for broadcast joins.

**Complexity:** where/select O(n); order_by O(n log n); group_by O(n); join O(n + m).

**Common mistakes**

- Returning `self` after mutating — breaks reuse of intermediate results.
- Nested-loop joins (O(n·m)).